In [ ]:
# ═══════════════════════════════════════════════════════════
# 1. IMPORTS
# ═══════════════════════════════════════════════════════════
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode, tools_condition

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain.chat_models import init_chat_model

from pydantic import BaseModel, Field
from typing import Annotated, Optional
from langgraph.graph.message import add_messages

# ═══════════════════════════════════════════════════════════
# 2. STATE DEFINITION (Use Pydantic BaseModel)
# ═══════════════════════════════════════════════════════════
class AppState(BaseModel):
    """Production state with validation"""
    messages: Annotated[list, add_messages]
    # Add custom fields as needed
    user_id: Optional[str] = None
    context: Optional[dict] = Field(default_factory=dict)
    
# ═══════════════════════════════════════════════════════════
# 3. TOOLS (Regular Python functions)
# ═══════════════════════════════════════════════════════════
def your_tool(param: str) -> str:
    """Tool docstring - LLM sees this"""
    return f"Result: {param}"

tools = [your_tool]

# ═══════════════════════════════════════════════════════════
# 4. LLM SETUP
# ═══════════════════════════════════════════════════════════
llm = init_chat_model(model="gpt-4")
llm_with_tools = llm.bind_tools(tools)

# ═══════════════════════════════════════════════════════════
# 5. NODES (Always use BaseModel attribute access: state.field)
# ═══════════════════════════════════════════════════════════
def agent_node(state: AppState) -> dict:
    """Main agent with system prompt"""
    system_msg = SystemMessage(content="You are a helpful assistant.")
    messages_with_system = [system_msg] + state.messages
    
    response = llm_with_tools.invoke(messages_with_system)
    return {"messages": [response]}

def custom_node(state: AppState) -> dict:
    """Custom processing node"""
    # Access state with dot notation
    last_msg = state.messages[-1]
    user_id = state.user_id
    
    # Return updates as dict
    return {
        "context": {"processed": True},
        "messages": [AIMessage(content="Processed")]
    }

# ═══════════════════════════════════════════════════════════
# 6. GRAPH CONSTRUCTION
# ═══════════════════════════════════════════════════════════
def build_graph():
    """Build and return compiled graph"""
    graph = StateGraph(AppState)
    
    # Add nodes
    graph.add_node("agent", agent_node)
    graph.add_node("tools", ToolNode(tools))
    
    # Add edges
    graph.add_edge(START, "agent")
    graph.add_conditional_edges("agent", tools_condition)
    graph.add_edge("tools", "agent")  # Loop back for multi-step
    
    # Compile with checkpointer for memory
    checkpointer = InMemorySaver()
    return graph.compile(checkpointer=checkpointer)

# ═══════════════════════════════════════════════════════════
# 7. INVOCATION (Always use HumanMessage, never dicts)
# ═══════════════════════════════════════════════════════════
app = build_graph()

# Single invocation
result = app.invoke({
    "messages": [HumanMessage(content="Your question here")]
})

# With persistence (thread_id for multi-turn)
config = {"configurable": {"thread_id": "user_123"}}
result = app.invoke(
    {"messages": [HumanMessage(content="Follow-up question")]},
    config=config
)

# Access result
final_answer = result["messages"][-1].content

# ═══════════════════════════════════════════════════════════
# 8. INTERACTIVE LOOP (Production pattern)
# ═══════════════════════════════════════════════════════════
def chat_loop(app):
    """Production-ready chat interface"""
    thread_id = input("Session ID: ")
    
    while True:
        user_input = input("\nYou: ")
        if user_input.lower() in ['quit', 'exit', 'q']:
            break
        
        result = app.invoke(
            {"messages": [HumanMessage(content=user_input)]},
            config={"configurable": {"thread_id": thread_id}}
        )
        
        print(f"AI: {result['messages'][-1].content}")